In [ ]:
# Final cleaned and analysis-ready dataset
print("="*60)
print("FINAL ANALYSIS-READY DATASET SUMMARY")
print("="*60)

print(f"\nShape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")
print(f"\nData Types:\n{df_clean.dtypes}")
print(f"\nMissing Values:\n{df_clean.isnull().sum()}")

print("\n" + "="*60)
print("KEY STATISTICS:")
print("="*60)
print(f"Total Customers: {len(df_clean)}")
print(f"Countries Represented: {df_clean['Country'].nunique()}")
print(f"Date Range: {df_clean['Subscription Date'].min().date()} to {df_clean['Subscription Date'].max().date()}")
print(f"Years Covered: {sorted(df_clean['Subscription Year'].unique())}")

print("\n" + "="*60)
print("Sample of cleaned data (first 5 rows):")
print("="*60)
display_cols = ['Customer Id', 'First Name', 'Last Name', 'Country', 'Subscription Date', 'Subscription Year']
print(df_clean[display_cols].head())

## 8. Final Summary - Analysis-Ready Dataset

In [ ]:
# Pivot Table 1: Subscriptions by Year and Month
pivot_year_month = pd.pivot_table(
    df_clean,
    values='Customer Id',
    index='Subscription Year',
    columns='Subscription Month',
    aggfunc='count',
    fill_value=0
)
print("Subscriptions by Year and Month (Pivot Table):")
print(pivot_year_month)

print("\n" + "="*60 + "\n")

# Pivot Table 2: Count of customers by Country and Year
pivot_country_year = pd.pivot_table(
    df_clean,
    values='Customer Id',
    index='Country',
    columns='Subscription Year',
    aggfunc='count',
    fill_value=0
)
print("Customers by Country and Subscription Year:")
print(pivot_country_year.sort_values(by=2021, ascending=False).head(10))

print("\n" + "="*60 + "\n")

# Pivot Table 3: With margins (totals)
pivot_with_margins = pd.pivot_table(
    df_merged[df_merged['Region'].notna()],  # Use merged data with Region
    values='Customer Id',
    index='Region',
    columns='Subscription Year',
    aggfunc='count',
    margins=True,
    fill_value=0
)
print("Customers by Region and Year (with totals):")
print(pivot_with_margins)

## 7. Pivot Tables

In [ ]:
# Create a sample "Country Regions" dataframe to demonstrate merging
country_regions = pd.DataFrame({
    'Country': ['Chile', 'Vietnam', 'Bosnia and Herzegovina', 'Bulgaria', 'Cyprus'],
    'Region': ['South America', 'Asia', 'Europe', 'Europe', 'Europe'],
    'Code': ['CL', 'VN', 'BA', 'BG', 'CY']
})

print("Country Regions Dataset:")
print(country_regions)

print("\n" + "="*60 + "\n")

# Merge customers with country regions
df_merged = pd.merge(df_clean, country_regions, on='Country', how='left')

print(f"Merged dataset shape: {df_merged.shape}")
print("\nMerged data sample:")
print(df_merged[['First Name', 'Country', 'Region']].head(10))

print("\n" + "="*60 + "\n")

# Check for customers from regions with data
print("Customers with Region information:")
print(df_merged[df_merged['Region'].notna()][['First Name', 'Country', 'Region']].head())

## 6. Merging Datasets

In [ ]:
# GroupBy 1: Customers by Country
customers_by_country = df_clean.groupby('Country').size().sort_values(ascending=False)
print("Customers by Country (Top 10):")
print(customers_by_country.head(10))

print("\n" + "="*60 + "\n")

# GroupBy 2: Multiple aggregations
country_stats = df_clean.groupby('Country').agg({
    'Customer Id': 'count',  # Total customers
    'Subscription Year': ['min', 'max'],  # Year range
    'City': 'nunique'  # Number of unique cities
}).round(2)
country_stats.columns = ['Total Customers', 'First Subscription', 'Latest Subscription', 'Unique Cities']
print("Country Statistics (Top 10):")
print(country_stats.sort_values('Total Customers', ascending=False).head(10))

print("\n" + "="*60 + "\n")

# GroupBy 3: Subscriptions by Year
subscriptions_by_year = df_clean.groupby('Subscription Year').agg({
    'Customer Id': 'count',
    'First Name': 'count'
}).rename(columns={'Customer Id': 'Number of Subscriptions', 'First Name': 'Customers'})
print("Subscriptions by Year:")
print(subscriptions_by_year)

## 5. GroupBy Operations

In [ ]:
# Filter 1: Customers from specific years
recent_customers = df_clean[df_clean['Subscription Year'] >= 2021]
print(f"Customers subscribed in 2021 or later: {len(recent_customers)}")
print(recent_customers[['First Name', 'Country', 'Subscription Date']].head())

print("\n" + "="*60 + "\n")

# Filter 2: Customers from specific countries
target_countries = ['Chile', 'Vietnam', 'Bosnia and Herzegovina']
customers_target_countries = df_clean[df_clean['Country'].isin(target_countries)]
print(f"Customers from {target_countries}:")
print(customers_target_countries[['First Name', 'Country']].head())

print("\n" + "="*60 + "\n")

# Filter 3: Customers with Email provided and from 2020
filtered_df = df_clean[(df_clean['Email'] != '') & (df_clean['Subscription Year'] == 2020)]
print(f"Customers from 2020 with email: {len(filtered_df)}")
print(filtered_df[['Customer Id', 'First Name', 'Subscription Date']].head())

## 4. Filtering Rows

In [ ]:
# Clean and standardize data
# 1. Remove leading/trailing whitespace from text columns
text_columns = ['First Name', 'Last Name', 'Company', 'City', 'Country']
df_clean[text_columns] = df_clean[text_columns].apply(lambda x: x.str.strip())

# 2. Remove duplicates based on Customer Id
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Customer Id'], keep='first')
print(f"Removed {initial_rows - len(df_clean)} duplicate records")

# 3. Create a subscription year column for analysis
df_clean['Subscription Year'] = df_clean['Subscription Date'].dt.year
df_clean['Subscription Month'] = df_clean['Subscription Date'].dt.month

print("\nDataset after cleaning:")
print(f"Shape: {df_clean.shape}")
print(f"\nFirst 3 rows of cleaned data:")
print(df_clean[['Customer Id', 'First Name', 'Last Name', 'Country', 'Subscription Date']].head(3))

## 3. Data Cleaning - Text & Format Standardization

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

# Introduce some missing values to simulate real-world messy data
np.random.seed(42)
missing_indices = np.random.choice(df_clean.index, size=10, replace=False)
df_clean.loc[missing_indices[:5], 'Phone 2'] = np.nan
df_clean.loc[missing_indices[5:], 'Email'] = np.nan

print("Missing values after introducing NaN:")
print(df_clean.isnull().sum())
print("\n" + "="*60)

# Handle missing values strategies
# 1. Drop rows where Email is missing (critical field)
df_clean = df_clean.dropna(subset=['Email'])

# 2. Fill missing Phone 2 with 'Not provided'
df_clean['Phone 2'] = df_clean['Phone 2'].fillna('Not provided')

# 3. Convert Subscription Date to datetime
df_clean['Subscription Date'] = pd.to_datetime(df_clean['Subscription Date'])

print("After handling missing values:")
print(df_clean.isnull().sum())
print(f"\nRows remaining: {len(df_clean)} (removed {len(df) - len(df_clean)} rows)")

## 2. Introduce and Handle Missing Values

In [ ]:
# Load the customers dataset
df = pd.read_csv('customers-100.csv')

print("Dataset Shape:", df.shape)
print("\n" + "="*60)
print("First 5 rows:")
print(df.head())
print("\n" + "="*60)
print("Data Types:")
print(df.dtypes)
print("\n" + "="*60)
print("Missing Values:")
print(df.isnull().sum())

## 1. Load and Explore the Dataset

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 30)

# Data Cleaning & Analysis - Customers Dataset

This notebook demonstrates essential data cleaning and manipulation techniques:
- Loading and exploring data
- Handling missing values
- Filtering rows
- GroupBy operations
- Merging datasets
- Creating pivot tables